# 🚢 Урок 15 — ООП на реальних даних: Пасажири Титаніка

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Навіщо цей урок?

На минулому уроці ми познайомились із класами через приклад `Dog` та `Robot`.
Ці приклади добре пояснюють *синтаксис*, але не показують, навіщо класи потрібні **в реальній роботі**.

Сьогодні ми зробимо крок уперед:

> 🎯 **Головна ідея уроку:**  
> Ми візьмемо реальний датасет — пасажирів Титаніка —  
> і перетворимо кожен рядок таблиці на **об'єкт із поведінкою**.

---

## Що ти навчишся робити

| Крок | Що робимо |
|------|-----------|
| 1 | Завантажуємо датасет і дивимось на сирі дані |
| 2 | Будуємо клас `Passenger` поступово, від простого до складного |
| 3 | Розбираємося з `self`, `__repr__`, методами |
| 4 | Вчимо різницю між атрибутами класу та екземпляра |
| 5 | Розуміємо namespace через `__dict__` |
| 6 | Будуємо `TitanicAnalyzer` для аналізу колекції об'єктів |

---

> 🐕 Абстрактні приклади (`Dog`, `Cat`) корисні для старту — вони короткі та зрозумілі.
> Але реальні дані показують, **чому ООП існує**: щоб перетворити сирий рядок таблиці
> на сутність із логікою та поведінкою.


---

## 🧭 Ментальна модель: від таблиці до об'єкта

Перед тим як писати код — уяви послідовність трансформацій:

```
DataFrame (таблиця)
   ↓
df.iloc[0]  →  рядок = pandas Series (сирі дані, просто числа і рядки)
   ↓
Passenger.from_series(row)  →  екземпляр класу (об'єкт із атрибутами та методами)
```

**Клас — це креслення пасажира.**  
**Екземпляр — це конкретний пасажир з конкретного рядка таблиці.**

| Поняття | Аналогія |
|---------|----------|
| `DataFrame` | Списки пасажирів на папері |
| Рядок | Один запис у списку |
| Клас `Passenger` | Анкета/форма, що описує поля |
| Екземпляр `p1` | Заповнена анкета конкретної людини |


---

## 📦 Крок 1: Завантажуємо датасет Titanic

Seaborn — це бібліотека для візуалізації, яка також надає кілька вбудованих датасетів для навчання.  
Titanic — один із найвідоміших: дані про пасажирів, які були (або не були) на борту 15 квітня 1912 року.


In [ ]:
import seaborn as sns
import pandas as pd

df = sns.load_dataset('titanic')
df.head(10)


In [ ]:
# Зберігаємо датасет у CSV файл
df.to_csv("titanic.csv", index=False)

**Колонки, які нас цікавлять:**

| Колонка | Що означає |
|---------|------------|
| `survived` | 1 = вижив, 0 = загинув |
| `pclass` | Клас квитка: 1 (перший), 2, 3 (третій) |
| `sex` | Стать: `male` / `female` |
| `age` | Вік (може бути `NaN`) |
| `fare` | Ціна квитка |
| `embarked` | Порт посадки: S, C або Q |
| `alone` | `True` якщо подорожував один |
| `sibsp` | Кількість братів/сестер або подружжя |
| `parch` | Кількість батьків/дітей |


In [ ]:
print('Рядків:', len(df))
print('Пропущені значення age:', df['age'].isna().sum())
print('Пропущені значення embarked:', df['embarked'].isna().sum())


In [ ]:
with open("titanic.csv", "r", encoding="utf-8") as f:
    content = f.read()

print(content[:500])  # щоб не вивалювати весь файл

In [ ]:
with open("titanic.csv", "r", encoding="utf-8") as f:
    df_from_file = pd.read_csv(f)

df_from_file.head()

In [ ]:
row = df_from_file.iloc[0]
print(row.to_dict())

---

## 🔬 Крок 2: Один рядок — це кандидат на об'єкт

Давай подивимось на першого пасажира:


In [ ]:
row = df.iloc[0]
print(type(row))
print(row)


Ось що ми бачимо: `pandas.Series` — це просто набір ключів та значень.  
Дані є, але **немає жодної поведінки**: ми не можемо запитати `row.is_child()` або `row.survived_text()`.

**Ось навіщо потрібен клас:**  
Ми хочемо взяти ці сирі дані і загорнути їх у об'єкт, який:

- **зберігає** атрибути (стан),
- **вміє** щось робити (методи).


---

## 🏗️ Крок 3: Перша версія класу `Passenger`

Почнемо з мінімуму — лише `__init__`.  
Жодних методів, просто зберігаємо дані у атрибутах екземпляра.

**Запитання перед кодом:** що виведе `p1.sex`? А `p2.age`?


In [ ]:
class Passenger:
    def __init__(self, sex, age, pclass, fare, survived):
        self.sex = sex
        self.age = age
        self.pclass = pclass
        self.fare = fare
        self.survived = survived


p1 = Passenger('male', 22, 3, 7.25, 0)
p2 = Passenger('female', 38, 1, 71.28, 1)

print(p1.sex, p1.age)
print(p2.sex, p2.age)
print(type(p1))


**Що ми щойно побачили:**

- `Passenger` — клас (blueprint)
- `p1` і `p2` — два різні екземпляри, кожен зі своїм набором даних
- Атрибути `sex`, `age` тощо живуть у `p1.__dict__` (простір імен екземпляра)

Перевіримо:


In [ ]:
print(p1.__dict__)
print(p2.__dict__)
print(p1)
print(p2)


---

## 🖨️ Крок 4: `__repr__` — щоб об'єкт умів себе показати

Зараз якщо просто ввести `p1` у консолі, ви побачите щось на кшталт:
```
<__main__.Passenger object at 0x7f...>
```
Це адреса в пам'яті — не дуже інформативно.

`__repr__` — спеціальний метод, який повертає **технічне представлення** об'єкта.  
Він викликається, коли ви вводите об'єкт у консолі або викликаєте `repr(obj)`.


In [ ]:
class Passenger:
    def __init__(self, sex, age, pclass, fare, survived):
        self.sex = sex
        self.age = age
        self.pclass = pclass
        self.fare = fare
        self.survived = survived

    def __repr__(self):
        status = 'survived' if self.survived == 1 else 'perished'
        return (
            f'Passenger(sex={self.sex!r}, age={self.age}, '
            f'class={self.pclass}, fare={self.fare:.2f}, status={status})'
        )


p1 = Passenger('male', 22, 3, 7.25, 0)
p2 = Passenger('female', 38, 1, 71.28, 1)

print(p1)
print(p2)


Тепер об'єкт «говорить» — одразу видно, хто це і що з ним сталося.  
Це не просто краса: при дебагу десятків об'єктів зрозумілий `__repr__` рятує.


---

## ⚙️ Крок 5: Методи екземпляра — поведінка об'єкта

Клас без методів — це просто «розумний словник».  
Справжня сила — у методах: вони дають об'єкту **поведінку**.

Додамо кілька корисних методів. Зверніть увагу: кожен метод приймає `self` —  
це посилання на **той конкретний об'єкт**, з яким ми зараз працюємо.

```
p1.is_child()
# Python робить це:
Passenger.is_child(p1)   # ← p1 передається як self
```


In [ ]:
class Passenger:
    def __init__(self, sex, age, pclass, fare, survived, sibsp=0, parch=0):
        self.sex = sex
        self.age = age
        self.pclass = pclass
        self.fare = fare
        self.survived = survived
        self.sibsp = sibsp
        self.parch = parch

    def __repr__(self):
        status = 'survived' if self.survived == 1 else 'perished'
        return (
            f'Passenger(sex={self.sex!r}, age={self.age}, '
            f'class={self.pclass}, fare={self.fare:.2f}, status={status})'
        )

    def is_child(self):
        """True якщо вік менше 18 і вік відомий."""
        return self.age is not None and self.age < 18

    def survived_text(self):
        """Текстовий статус виживання."""
        return 'вижив(ла)' if self.survived == 1 else 'загинув(ла)'

    def ticket_level(self):
        """Рівень квитка за ціною."""
        if self.fare is None:
            return 'невідомо'
        if self.fare < 10:
            return 'economy'
        elif self.fare < 30:
            return 'standard'
        return 'premium'

    def family_size(self):
        """Загальний розмір сімейної групи на борту."""
        return self.sibsp + self.parch + 1


p1 = Passenger('male', 22, 3, 7.25, 0, sibsp=0, parch=0)
p2 = Passenger('female', 5, 3, 12.47, 1, sibsp=1, parch=2)

print(p1, '|', p1.survived_text(), '|', p1.ticket_level())
print(p2, '|', p2.survived_text(), '|', p2.ticket_level())
print('p2 — дитина?', p2.is_child())
print('p2 — розмір сім\'ї:', p2.family_size())


**Що ми щойно побачили:**

Кожен метод звертається до **своїх** даних через `self`.  
`p1.ticket_level()` і `p2.ticket_level()` — один і той самий метод,  
але `self` у першому випадку — це `p1`, у другому — `p2`.

Саме це і є суть `self`: він дає методу знати, *чиї* дані обробляти.


---

## 🏭 Крок 6: `@classmethod` — створення об'єкта з рядка DataFrame

Ми вже можемо створювати `Passenger` вручну, але це незручно:  
треба вручну вказувати кожен аргумент для кожного рядка.

**Кращий підхід:** альтернативний конструктор `from_series`,  
який приймає рядок DataFrame і сам витягує потрібні поля.

**Різниця між `self` і `cls`:**

| | `self` | `cls` |
|-|--------|-------|
| Що це | Конкретний екземпляр | Сам клас |
| Де використовується | У звичайних методах | У `@classmethod` |
| Навіщо | Доступ до даних об'єкта | Створення нового об'єкта |


In [ ]:
class Passenger:

    def __init__(self, sex, age, pclass, fare, survived, alone, embarked, sibsp=0, parch=0):
        self.sex = sex
        self.age = age
        self.pclass = pclass
        self.fare = fare
        self.survived = survived
        self.alone = alone
        self.embarked = embarked
        self.sibsp = sibsp
        self.parch = parch

    def __repr__(self):
        status = 'survived' if self.survived == 1 else 'perished'
        return (
            f'Passenger(sex={self.sex!r}, age={self.age}, '
            f'class={self.pclass}, status={status})'
        )

    def is_child(self):
        return self.age is not None and self.age < 18

    def survived_text(self):
        return 'вижив(ла)' if self.survived == 1 else 'загинув(ла)'

    def ticket_level(self):
        if self.fare is None:
            return 'невідомо'
        if self.fare < 10:
            return 'economy'
        elif self.fare < 30:
            return 'standard'
        return 'premium'

    def family_size(self):
        return self.sibsp + self.parch + 1

    @classmethod
    def from_series(cls, row):
        return cls(
            sex=row['sex'],
            age=row['age'] if not pd.isna(row['age']) else None,
            pclass=row['pclass'],
            fare=row['fare'],
            survived=row['survived'],
            alone=row['alone'],
            embarked=row['embarked'],
            sibsp=row['sibsp'],
            parch=row['parch'],
        )

    @classmethod
    def from_dict(cls, data):
        return cls(
            sex=data['sex'],
            age=data['age'] if not pd.isna(data['age']) else None,
            pclass=data['pclass'],
            fare=data['fare'],
            survived=data['survived'],
            alone=data['alone'],
            embarked=data['embarked'],
            sibsp=data['sibsp'],
            parch=data['parch'],
        )



In [ ]:
# Тепер можемо легко створити об'єкт із будь-якого рядка
p1 = Passenger.from_series(df.iloc[0])
p2 = Passenger.from_series(df.iloc[1])

print(p1)
print(p2)
print('p1 — рівень квитка:', p1.ticket_level())
print('p2 —', p2.survived_text())


In [ ]:
# через pandas
p1 = Passenger.from_series(df.iloc[0])

# через dict
row_dict = df.iloc[1].to_dict()
p2 = Passenger.from_dict(row_dict)

print(p1)
print(p2)

**Як це працює:**

```python
Passenger.from_series(df.iloc[0])
```

1. `cls` — це сам клас `Passenger`
2. Метод витягує потрібні поля з рядка
3. Викликає `cls(...)` — тобто `Passenger(...)` — і повертає готовий об'єкт

Тепер рядок DataFrame трансформується в об'єкт за один рядок коду.


---

## 🏷️ Крок 7: Атрибути класу — спільні для всіх

До цього ми додавали лише **атрибути екземпляра** — індивідуальні для кожного пасажира.  
Але є ще **атрибути класу** — спільні для всіх екземплярів.

Коли їх використовувати:

- **Константи** для всього класу (`allowed_embark_ports`)
- **Лічильники** або метадані про клас (`count`)
- **Характеристики**, що стосуються всіх екземплярів (`species`)


In [ ]:
class Passenger:
    # --- Атрибути КЛАСУ ---
    species = 'Homo sapiens'
    allowed_embark_ports = ('S', 'C', 'Q')
    count = 0

    def __init__(self, sex, age, pclass, fare, survived, alone, embarked, sibsp=0, parch=0):
        self.sex = sex
        self.age = age
        self.pclass = pclass
        self.fare = fare
        self.survived = survived
        self.alone = alone
        self.embarked = embarked
        self.sibsp = sibsp
        self.parch = parch
        Passenger.count += 1

    def __repr__(self):
        status = 'survived' if self.survived == 1 else 'perished'
        return (
            f'Passenger(sex={self.sex!r}, age={self.age}, '
            f'class={self.pclass}, status={status})'
        )

    def is_child(self):
        return self.age is not None and self.age < 18

    def survived_text(self):
        return 'вижив(ла)' if self.survived == 1 else 'загинув(ла)'

    def ticket_level(self):
        if self.fare is None:
            return 'невідомо'
        if self.fare < 10:
            return 'economy'
        elif self.fare < 30:
            return 'standard'
        return 'premium'

    def family_size(self):
        return self.sibsp + self.parch + 1

    #  ГОЛОВНИЙ КОНСТРУКТОР (через dict)
    @classmethod
    def from_dict(cls, data):
        return cls(
            sex=data.get('sex'),
            age=data.get('age') if not pd.isna(data.get('age')) else None,
            pclass=data.get('pclass'),
            fare=data.get('fare'),
            survived=data.get('survived'),
            alone=data.get('alone'),
            embarked=data.get('embarked'),
            sibsp=data.get('sibsp', 0),
            parch=data.get('parch', 0),
        )

    #  ТЕПЕР Series = просто dict
    @classmethod
    def from_series(cls, row):
        return cls.from_dict(row.to_dict())

    @staticmethod
    def is_valid_port(port):
        return port in {'S', 'C', 'Q'}

In [ ]:

p1 = Passenger.from_series(df.iloc[0])
p2 = Passenger.from_series(df.iloc[1])
p3 = Passenger.from_series(df.iloc[2])
p4 = Passenger.from_series(df.iloc[3])

print('Вид:', Passenger.species)           # доступно через клас
print('Вид через p1:', p1.species)         # і через екземпляр (Python шукає в класі)
print('Дозволені порти:', Passenger.allowed_embark_ports)
print('Створено об\'єктів:', Passenger.count)


---

## 🗃️ Крок 8: Namespace — де живуть атрибути?

У Python кожен об'єкт і кожен клас має свій **словник атрибутів** — `__dict__`.  
Це і є namespace.

Правило пошуку атрибута `p1.x`:
```
1. Шукаємо в p1.__dict__    →  знайдено? Повертаємо.
2. Шукаємо в Passenger.__dict__  →  знайдено? Повертаємо.
3. Шукаємо в батьківських класах (object...)
4. AttributeError
```


In [ ]:
# Дивимось на словники
print('p1.__dict__:')
print(p1.__dict__)
print()
print('Passenger.__dict__ ключі:')
print(list(Passenger.__dict__.keys()))
print(list(Passenger.__dir__(p1)))
print(Passenger.__class__(Passenger))



Зверніть увагу: у `p1.__dict__` — **лише особисті** атрибути пасажира.  
`species`, `count`, методи — вони живуть у `Passenger.__dict__`, не у `p1`.

### 🔍 Що відбувається при затінені атрибута?

Ось класична пастка: здається, що ми змінили атрибут класу, але насправді — ні.


In [ ]:
print('До зміни:')
print('p1.species:', p1.species)       # береться з Passenger.__dict__
print('p2.species:', p2.species)       # теж

# Присвоєння через екземпляр → створює ОСОБИСТИЙ атрибут у p1
p1.species = 'Пасажир (особливий)'

print()
print('Після p1.species = ...')
print('p1.species:', p1.species)       # тепер читається з p1.__dict__
print('p2.species:', p2.species)       # p2 не змінився — береться з класу
print('Passenger.species:', Passenger.species)  # клас не змінився!

print()
print('p1.__dict__:', p1.__dict__)     # тепер у p1 є власний 'species'


> ⚠️ **Важливо!**  
> `p1.species = 'Пасажир (особливий)'` — це **не зміна класу**.  
> Це створення **нового атрибута в `p1.__dict__`**, який **затінює** класовий атрибут.  
> `p2` і `Passenger` залишились незачеплені.

Щоб змінити атрибут класу — треба звертатися саме до класу:


In [ ]:
Passenger.species = 'Homo sapiens (оновлено)'
print(p2.species)   # бачить зміну класового атрибута


---

## 🔧 Крок 9: `@staticmethod` — логіка без `self` і `cls`

Іноді потрібна функція, яка логічно пов'язана з класом, але **не потребує доступу** ні до екземпляра, ні до класу.  
Для цього є `@staticmethod`.

Ми вже додали `is_valid_port` у клас. Подивимось, як він працює:


In [ ]:
# Викликається і через клас, і через екземпляр
print(Passenger.is_valid_port('S'))   # True
print(Passenger.is_valid_port('X'))   # False
print(p1.is_valid_port(p1.embarked))  # перевіряємо порт конкретного пасажира


**Навіщо `@staticmethod`, якщо можна зробити звичайну функцію?**

🔧 `@staticmethod` — де має жити логіка?

Коли ми пишемо код, важливе питання не тільки **"як написати функцію"**, а:

>  **Де ця логіка повинна жити?**

---

###  Є 3 типи логіки в класах

#### 1️⃣ Логіка про КОНКРЕТНИЙ об'єкт

```python
p1.ticket_level()
````

* використовує дані пасажира (`self.fare`)
* залежить від конкретного екземпляра

👉 Це **звичайний метод (`self`)**

---

#### 2️⃣ Логіка про КЛАС

```python
Passenger.from_series(row)
```

* створює об'єкт
* працює з самим класом

👉 Це **`@classmethod` (`cls`)**

---

#### 3️⃣ Логіка "ПРОСТО ПРО ТЕМУ"

```python
Passenger.is_valid_port('S')
```

* не використовує ні `self`, ні `cls`
* це просто правило

👉 Це **`@staticmethod`**

---

###  Що таке `@staticmethod`?

>  **Це звичайна функція, але вона логічно належить класу**

Вона:

* не залежить від конкретного об'єкта
* не залежить від класу
* але має сенс саме в контексті цього класу

---

### ❓ Можна без нього?

Так, можна написати так:

```python
def is_valid_port(port):
    return port in {'S', 'C', 'Q'}
```

Але тоді виникає проблема:

👉 де ця функція "живе"?

---

### ❌ Поганий варіант (хаос)

```python
is_valid_port()
calculate_ticket()
check_child()
format_status()
```

* все в глобальному просторі
* незрозуміло, до чого це відноситься

---

### ✅ Хороший варіант (структура)

```python
Passenger.is_valid_port()
Passenger.some_other_logic()
```

* логіка згрупована
* код читається як мова предметної області

---

###  Головна ідея

```text
staticmethod = це не про "як працює код"
staticmethod = це про "як організований код"
```

---

### ⚡ Як швидко визначити, що використовувати?

```text
Використовує self? → звичайний метод
Використовує cls?  → classmethod
Не використовує нічого? → staticmethod або функція
```

👉 Далі ключове питання:

```text
Чи логічно ця функція належить класу?
```

* Так → `@staticmethod`
* Ні → звичайна функція

---

###  Найчесніше визначення

> `@staticmethod` — це просто функція, покладена в клас для порядку.

---

###  Висновок

Ми використовуємо `@staticmethod`, коли хочемо:

* зберегти **чисту структуру коду**
* згрупувати логіку за змістом
* зробити код більш читабельним і логічним

Це вже не про синтаксис.

👉 Це про **архітектуру мислення**.


| Тип | Перший аргумент | Використання |
|-----|----------------|---------------|
| Звичайний метод | `self` (екземпляр) | Доступ до даних конкретного об'єкта |
| `@classmethod` | `cls` (клас) | Альтернативний конструктор, робота з класом |
| `@staticmethod` | нічого | Утилітарна логіка, пов'язана з темою класу |


---

## 🔭 Крок 10: LEGB та класовий namespace

Є одна важлива деталь, яка часто плутає новачків:

> **Простір імен класу не є частиною LEGB для методів.**

Нагадаємо правило пошуку змінних у Python — LEGB:
```
L — Local     (всередині методу)
E — Enclosing (охоплююча функція, для вкладених функцій)
G — Global    (рівень модуля)
B — Built-in  (print, len...)
```

Клас `Passenger` і його атрибути — **не входять** у цей ланцюг!  
Від всередині методу, щоб дістатися до `species`, потрібен явний кваліфікатор:


In [ ]:
status = 'GLOBAL'   # глобальна змінна

class DemoPassenger:
    status = 'CLASS'   # атрибут класу (НЕ є enclosing для методів)

    def show(self):
        status = 'LOCAL'          # локальна змінна
        print('local:', status)   # знаходить LOCAL
        print('self:', self.status)   # знаходить CLASS через атрибутний доступ

    def show_without_local(self):
        # Тут немає локального 'status'
        print(status)             # перестрибує клас → знаходить GLOBAL!
        print(self.status)        # а тут знаходить CLASS


d = DemoPassenger()
print('--- show() ---')
d.show()
print('--- show_without_local() ---')
d.show_without_local()


> 💡 **Правило:**  
> Щоб звернутися до атрибута класу всередині методу — завжди пишіть `self.x` або `ClassName.x`.  
> Без кваліфікатора Python шукає за LEGB і **пропускає** простір імен класу.


---

## 👥 Крок 11: Список пасажирів — від рядків до об'єктів

Тепер застосуємо все разом: перетворимо перші 20 рядків DataFrame на список об'єктів.


In [ ]:
passengers = [
    Passenger.from_dict(row.to_dict())
    for _, row in df.head(20).iterrows()
]

print(f'Створено {len(passengers)} об\'єктів. Загальний лічильник: {Passenger.count}')
print()

for p in passengers:
    child_marker = ' 👶' if p.is_child() else ''
    skull_marker = ' ☠️' if p.survived == 0 else ''

    print(f'{p}  |  {p.ticket_level():8}  |  {p.survived_text()}{child_marker}{skull_marker}')


**Що ми щойно побачили:**

Ми більше не працюємо з голими рядками типу `df.iloc[i]['survived']`.  
Кожен пасажир — це **об'єкт** із зрозумілими методами: `survived_text()`, `ticket_level()`, `is_child()`.  

Саме в цьому і є цінність ООП: код стає читабельнішим і ближчим до предметної області.


---

## 📊 Крок 12: Клас `TitanicAnalyzer` — аналіз колекції

Клас `Passenger` описує **одну сутність**.  
Але нам ще потрібна логіка для роботи з **колекцією** пасажирів.

Для цього ми створимо окремий клас `TitanicAnalyzer`.  
Це класичний архітектурний поділ:

| Клас | Роль |
|------|------|
| `Passenger` | Сутність (entity): дані + поведінка одного пасажира |
| `TitanicAnalyzer` | Сервіс (service): аналіз колекції пасажирів |


In [ ]:
class TitanicAnalyzer:
    def __init__(self, passengers):
        self.passengers = passengers

    def survival_rate(self):
        """Відсоток тих, хто вижив."""
        survived = [p for p in self.passengers if p.survived == 1]
        return len(survived) / len(self.passengers)

    def survived_passengers(self):
        """Список тих, хто вижив."""
        return [p for p in self.passengers if p.survived == 1]

    def non_survivors(self):
        """Список тих, хто загинув."""
        return [p for p in self.passengers if p.survived == 0]

    def children(self):
        """Список дітей."""
        return [p for p in self.passengers if p.is_child()]

    def average_fare(self):
        """Середня ціна квитка."""
        fares = [p.fare for p in self.passengers if p.fare is not None]
        return sum(fares) / len(fares) if fares else 0

    def survival_by_class(self):
        """Словник: клас квитка → відсоток виживання."""
        result = {}
        for pclass in [1, 2, 3]:
            group = [p for p in self.passengers if p.pclass == pclass]
            if group:
                survived = [p for p in group if p.survived == 1]
                result[pclass] = len(survived) / len(group)
        return result

    def women_who_survived(self):
        """Список жінок, які вижили."""
        return [
            p for p in self.passengers
            if p.sex == 'female' and p.survived == 1
        ]


---

## 🔍 Крок 13: Фінальний аналіз — Хто вижив?

Тепер завантажимо **всіх** пасажирів і запустимо аналіз.


In [ ]:
# Створюємо об'єкти для всього датасету через dict
all_passengers = [
    Passenger.from_dict(row.to_dict())
    for _, row in df.iterrows()
]

analyzer = TitanicAnalyzer(all_passengers)

print(f'Всього пасажирів: {len(all_passengers)}')
print(f'Загальний лічильник Passenger.count: {Passenger.count}')

In [ ]:
rate = analyzer.survival_rate()
survived = analyzer.survived_passengers()
lost = analyzer.non_survivors()

print(f'Загальний рівень виживання: {rate:.1%}')
print(f'Вижило: {len(survived)} людей')
print(f'Загинуло: {len(lost)} людей')


In [ ]:
by_class = analyzer.survival_by_class()
print('Виживання за класом квитка:')
for pclass, rate in sorted(by_class.items()):
    bar = '█' * int(rate * 20)
    print(f'  Клас {pclass}: {rate:.1%}  {bar}')


In [ ]:
print('Середня ціна квитка (всі):', f'{analyzer.average_fare():.2f}')

survived_fares = TitanicAnalyzer(survived).average_fare()
lost_fares = TitanicAnalyzer(lost).average_fare()
print(f'Середній квиток у тих, хто вижив:  {survived_fares:.2f}')
print(f'Середній квиток у тих, хто загинув: {lost_fares:.2f}')


In [ ]:
children = analyzer.children()
print(f'Дітей на борту: {len(children)}')
children_survived = [c for c in children if c.survived == 1]
if children:
    print(f'Виживання серед дітей: {len(children_survived)/len(children):.1%}')

print()
print('Жінки, що вижили (перші 5):')
for w in analyzer.women_who_survived()[:5]:
    print(' ', w)


In [ ]:
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# 1. Загальна статистика
# ─────────────────────────────────────────────

rate = analyzer.survival_rate()
survived = analyzer.survived_passengers()
lost = analyzer.non_survivors()

plt.figure(figsize=(6, 4))

plt.bar(['Survived', 'Lost'],
        [len(survived), len(lost)],
        color=['green', 'red'])

plt.title(f'Загальний рівень виживання: {rate:.1%}')
plt.ylabel('Кількість людей')

plt.show()



# ─────────────────────────────────────────────
# 2. Виживання по класах
# ─────────────────────────────────────────────

by_class = analyzer.survival_by_class()

classes = []
rates = []

for pclass, r in sorted(by_class.items()):
    classes.append(f'Class {pclass}')
    rates.append(r)

plt.figure(figsize=(6, 4))

plt.bar(classes, rates, color='blue')

plt.title('Виживання за класом')
plt.ylabel('Survival Rate')

plt.ylim(0, 1)

# підпис % над барами
for i, r in enumerate(rates):
    plt.text(i, r + 0.02, f'{r:.1%}', ha='center')

plt.show()


# ─────────────────────────────────────────────
# 3. Діти
# ─────────────────────────────────────────────

children = analyzer.children()
children_survived = [c for c in children if c.survived == 1]

if children:
    survived_count = len(children_survived)
    lost_count = len(children) - survived_count

    plt.figure(figsize=(5, 5))

    plt.pie(
        [survived_count, lost_count],
        labels=['Survived', 'Lost'],
        autopct='%1.1f%%',
        colors=['green', 'red']
    )

    plt.title('Виживання серед дітей')

    plt.show()


# ─────────────────────────────────────────────
# 4. Жінки (просто print залишимо)
# ─────────────────────────────────────────────

print('Жінки, що вижили (перші 5):')
for w in analyzer.women_who_survived()[:5]:
    print(' ', w)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


class TitanicAnalyzer:
    def __init__(self, passengers):
        self.passengers = passengers

    def survival_rate(self):
        survived = [p for p in self.passengers if p.survived == 1]
        return len(survived) / len(self.passengers) if self.passengers else 0

    def survived_passengers(self):
        return [p for p in self.passengers if p.survived == 1]

    def non_survivors(self):
        return [p for p in self.passengers if p.survived == 0]

    def children(self):
        return [p for p in self.passengers if p.is_child()]

    def average_fare(self):
        fares = [p.fare for p in self.passengers if p.fare is not None]
        return sum(fares) / len(fares) if fares else 0

    def survival_by_class(self):
        result = {}
        for pclass in [1, 2, 3]:
            group = [p for p in self.passengers if p.pclass == pclass]
            if group:
                survived = [p for p in group if p.survived == 1]
                result[pclass] = len(survived) / len(group)
        return result

    def women_who_survived(self):
        return [
            p for p in self.passengers
            if p.sex == 'female' and p.survived == 1
        ]

    def to_dataframe(self):
        rows = []
        for p in self.passengers:
            rows.append({
                "survived": p.survived,
                "pclass": p.pclass,
                "sex": p.sex,
                "age": p.age,
                "fare": p.fare,
                "embarked": getattr(p, "embarked", None),
                "is_child": p.is_child(),
            })
        return pd.DataFrame(rows)

    def plot_survival_hist(self):
        df = self.to_dataframe()

        plt.figure(figsize=(7, 4))
        sns.histplot(data=df, x="survived", discrete=True, shrink=0.6)
        plt.title("Розподіл: вижили / загинули")
        plt.xlabel("survived (0 = no, 1 = yes)")
        plt.ylabel("Кількість пасажирів")
        plt.show()

    def plot_fare_distribution(self):
        df = self.to_dataframe()

        plt.figure(figsize=(8, 4))
        sns.histplot(data=df, x="fare", bins=30, kde=True)
        plt.title("Розподіл вартості квитка")
        plt.xlabel("Fare")
        plt.ylabel("Кількість пасажирів")
        plt.show()

    def plot_age_distribution_by_survival(self):
        df = self.to_dataframe()

        plt.figure(figsize=(8, 4))
        sns.histplot(
            data=df,
            x="age",
            hue="survived",
            bins=25,
            kde=True,
            multiple="stack"
        )
        plt.title("Вік пасажирів і виживання")
        plt.xlabel("Age")
        plt.ylabel("Кількість")
        plt.show()

    def plot_fare_vs_age(self):
        df = self.to_dataframe()

        plt.figure(figsize=(8, 5))
        sns.scatterplot(
            data=df,
            x="age",
            y="fare",
            hue="survived",
            style="sex",
            size="pclass"
        )
        plt.title("Age vs Fare")
        plt.xlabel("Age")
        plt.ylabel("Fare")
        plt.show()

    def plot_fare_vs_age_by_class(self):
        df = self.to_dataframe()

        plt.figure(figsize=(8, 5))
        sns.scatterplot(
            data=df,
            x="age",
            y="fare",
            hue="pclass",
            style="survived",
            palette="deep"
        )
        plt.title("Age vs Fare by class")
        plt.xlabel("Age")
        plt.ylabel("Fare")
        plt.show()

    def plot_children_analysis(self):
        df = self.to_dataframe()
        children_df = df[df["is_child"] == True]

        if children_df.empty:
            print("Дітей у даних немає")
            return

        plt.figure(figsize=(7, 4))
        sns.histplot(
            data=children_df,
            x="survived",
            discrete=True,
            shrink=0.6
        )
        plt.title("Виживання серед дітей")
        plt.xlabel("survived")
        plt.ylabel("Кількість дітей")
        plt.show()

    def plot_survival_by_class_hist(self):
        df = self.to_dataframe()

        plt.figure(figsize=(7, 4))
        sns.histplot(
            data=df,
            x="pclass",
            hue="survived",
            multiple="dodge",
            discrete=True,
            shrink=0.7
        )
        plt.title("Виживання за класом квитка")
        plt.xlabel("Pclass")
        plt.ylabel("Кількість пасажирів")
        plt.show()

    def plot_women_survivors_fare_age(self):
        df = self.to_dataframe()
        women_df = df[(df["sex"] == "female") & (df["survived"] == 1)]

        if women_df.empty:
            print("Жінок, які вижили, немає")
            return

        plt.figure(figsize=(8, 5))
        sns.scatterplot(
            data=women_df,
            x="age",
            y="fare",
            hue="pclass",
            size="fare"
        )
        plt.title("Жінки, які вижили: Age vs Fare")
        plt.xlabel("Age")
        plt.ylabel("Fare")
        plt.show()

In [ ]:
all_passengers = [
    Passenger.from_dict(row.to_dict())
    for _, row in df.iterrows()
]

analyzer = TitanicAnalyzer(all_passengers)

analyzer.plot_survival_hist()
analyzer.plot_fare_distribution()
analyzer.plot_age_distribution_by_survival()
analyzer.plot_fare_vs_age()
analyzer.plot_fare_vs_age_by_class()
analyzer.plot_children_analysis()
analyzer.plot_survival_by_class_hist()
analyzer.plot_women_survivors_fare_age()

---

## 📋 Підсумок уроку

Ми пройшли шлях від сирого рядка DataFrame до повноцінної об'єктної моделі.

| Концепція | Що це | Як використовували |
|-----------|-------|--------------------|
| **Клас** | Шаблон/blueprint | `class Passenger:` |
| **Екземпляр** | Конкретний об'єкт | `p1 = Passenger.from_series(row)` |
| **`self`** | Посилання на екземпляр | `self.sex`, `self.age` у методах |
| **`__init__`** | Ініціалізатор | Наповнює об'єкт даними при створенні |
| **`__repr__`** | Текстове представлення | `print(p1)` виводить читабельний рядок |
| **Методи екземпляра** | Поведінка об'єкта | `is_child()`, `ticket_level()` |
| **Атрибути класу** | Спільні для всіх | `species`, `count`, `allowed_embark_ports` |
| **Атрибути екземпляра** | Індивідуальні | `self.sex`, `self.age` |
| **Namespace** | `__dict__` | `p1.__dict__` vs `Passenger.__dict__` |
| **`@classmethod`** | Альтернативний конструктор | `Passenger.from_series(row)` |
| **`@staticmethod`** | Утиліта без self/cls | `Passenger.is_valid_port('S')` |

---

### Коли використовувати DataFrame, а коли — клас?

| Задача | Що краще |
|--------|----------|
| Фільтрація, агрегація, статистика | `pandas DataFrame` |
| Складна бізнес-логіка для одної сутності | Клас |
| Читабельний код з іменованою поведінкою | Клас |
| Швидкий EDA (exploratory data analysis) | `pandas DataFrame` |

На практиці вони **доповнюють одне одного**: DataFrame для масових операцій,  
класи — коли потрібна модель предметної області.


---

## 🛠️ Спробуй сам — 5 практичних завдань

### Завдання 1: Метод `is_adult()`

Додайте до класу `Passenger` метод `is_adult()`, який повертає `True`, якщо вік пасажира ≥ 18.
Врахуйте випадок, коли `age is None`.


In [ ]:
# Завдання 1
# Ваш код:

# class Passenger:
#     ...
#     def is_adult(self):
#         pass

# Тест:
# p = Passenger.from_series(df.iloc[0])
# print(p.is_adult())


### Завдання 2: Метод `fare_per_family_member()`

Додайте метод, який повертає ціну квитка, поділену на розмір сімейної групи (`family_size()`).  
Це дає уявлення про реальну вартість поїздки на одну людину.


In [ ]:
# Завдання 2
# Ваш код:


### Завдання 3: `@staticmethod` для валідації класу квитка

Додайте статичний метод `is_valid_pclass(pclass)`, який перевіряє,  
чи є клас квитка допустимим (1, 2 або 3).

Використайте його в `__init__` для валідації.


In [ ]:
# Завдання 3
# Ваш код:


### Завдання 4: Виживання серед дітей у `TitanicAnalyzer`

Додайте до `TitanicAnalyzer` метод `child_survival_rate()`,  
який повертає відсоток виживання серед пасажирів-дітей (вік < 18).


In [ ]:
# Завдання 4
# Ваш код:


### Завдання 5: Топ-5 найдорожчих квитків серед тих, хто вижив

Використовуючи `analyzer.survived_passengers()`,  
знайдіть 5 пасажирів з найдорожчими квитками серед тих, хто вижив.

Підказка: використайте `sorted()` з аргументом `key`.


In [ ]:
# Завдання 5
# Ваш код:


## Клас — це не про код

Ти щойно попрацював із класами, методами і об'єктами.

Але найважливіше — не синтаксис.

>  **Клас — це не про код.
> Клас — це про модель реальності.**

---

### Що це означає?

У програмуванні ми завжди працюємо з даними.

Але дані самі по собі — це просто значення:

```text
age = 22
fare = 7.25
status = 0
````

- 👉 вони нічого не “означають”, поки ми не дамо їм контекст

---

###  І тут з’являється клас

```python
class Passenger:
```

- 👉 ми більше не працюємо з окремими змінними

- 👉 ми описуємо **сутність**

---

### Зміна мислення

Було:

```text
"є набір змінних"
```

Стало:

```text
"є об'єкт, який представляє щось у світі"
```

---

###  Що дає клас?

Клас дозволяє:

* об’єднати дані в одну структуру
* додати до цих даних поведінку
* працювати з об’єктом як з цілісною одиницею

---

###  Чому це важливо?

Бо в реальному світі ми мислимо не змінними, а об’єктами:

```text
не "age + fare"
а "людина"
```

```text
не "x, y координати"
а "точка"
```

```text
не "сума і товари"
а "замовлення"
```

---

###  І код починає це відображати

```python
user.login()
order.total_price()
point.distance_to(other)
```

- 👉 код стає ближчим до мислення
- 👉 і його легше читати, розуміти і розширювати

---

###  Глибший рівень

Коли ти використовуєш класи, ти відповідаєш на питання:

```text
"Які об'єкти існують у моїй системі?"
```

і

```text
"Що вони вміють робити?"
```

---

### Найважливіший інсайт

```text
Клас — це спосіб описати світ у коді
```

---

###  Висновок

> Python дозволяє писати код.
> Але класи дозволяють будувати **структуру і сенс**.

## **І саме це ти щойно почав робити.**
